# Nawy Properties Scraper
API: https://listing-api.nawy.com/v1/search/properties

Fetches all ~16,707 properties and saves to nawy_properties.csv


In [ ]:
import requests
import csv
import time
import sys
from datetime import datetime

# ── Config ──────────────────────────────────────────────────────────────────
API_URL    = "https://listing-api.nawy.com/v1/search/properties"
PAGE_SIZE  = 50          # max out the page size to reduce total requests
OUTPUT_CSV = "nawy_properties_full.csv"
DELAY      = 0.3         # seconds between requests (be polite to the server)
MAX_RETRIES = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; NawyDataCollector/1.0)",
    "Accept": "application/json",
}

# ── CSV columns ──────────────────────────────────────────────────────────────
FIELDNAMES = [
    "ID", "Title", "Subtitle", "Property_Type",
    "Finishing", "Area", "Beds", "Baths",
    "Has_Offer", "Sale_Type",
    "Compound", "Location", "Developer",
    "Price", "Down_Payment", "Installment",
    "Payment_Years", "Is_Cash","Ready_By",
    "Has_Mortgage",
    "Nawy_Selection",
    "Down_Payment_Pct",
    "Payment_Frequency",
]

def fetch_page(session: requests.Session, page: int) -> dict:
    """Fetch a single page with retry logic."""
    params = {"page": page, "pageSize": PAGE_SIZE}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(API_URL, params=params, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, ValueError) as e:
            if attempt == MAX_RETRIES:
                raise
            wait = attempt * 2
            print(f"  Page {page} attempt {attempt} failed ({e}). Retrying in {wait}s …")
            time.sleep(wait)


def parse_property(prop: dict) -> dict:
    """Extract the required fields from a single property record."""
    payment   = prop.get("paymentPlan") or {}
    compound  = prop.get("compound")    or {}
    area      = prop.get("area")        or {}
    developer = prop.get("developer")   or {}

    return {
        "ID":            prop.get("id", ""),
        "Title":            prop.get("title", ""),
        "Subtitle":         prop.get("subtitle", ""),
        "Property_Type":    prop.get("propertyType", ""),
        "Finishing":        prop.get("finishing", ""),
        "Area":        prop.get("unitArea", ""),
        "Beds":             prop.get("numberOfBedrooms", ""),
        "Baths":            prop.get("numberOfBathrooms", ""),
        "Has_Offer":        prop.get("hasOffer", ""),
        "Sale_Type":        prop.get("saleType", ""),
        # Nested
        "Compound":         (prop.get("compound") or {}).get("name", ""),
        "Location":         (prop.get("area") or {}).get("name", ""),
        "Developer":        (prop.get("developer") or {}).get("name", ""),
        "Price":      (prop.get("paymentPlan") or {}).get("minPrice", ""),
        "Down_Payment":     (prop.get("paymentPlan") or {}).get("minDownPayment", ""),
        "Installment":      (prop.get("paymentPlan") or {}).get("minInstallment", ""),
        "Payment_Years":    (prop.get("paymentPlan") or {}).get("numberOfInstallmentYears", ""),
        "Is_Cash":          (prop.get("paymentPlan") or {}).get("isCash", ""),
        "Ready_By": (prop.get("readyBy") or "")[:10],
        "Has_Mortgage":         prop.get("hasMortgage", ""),
        "Nawy_Selection":       prop.get("nawySelection", ""),
        "Down_Payment_Pct":     (prop.get("paymentPlan") or {}).get("downPaymentPercentage", ""),
        "Payment_Frequency":    (prop.get("paymentPlan") or {}).get("frequency", ""),
    }


def scrape_all():
    start_time = datetime.now()
    print(f"    Nawy scraper started at {start_time.strftime('%H:%M:%S')}")
    print(f"    Output → {OUTPUT_CSV}\n")

    session = requests.Session()

    # ── First request: discover total count ──────────────────────────────────
    first = fetch_page(session, page=1)
    total      = first.get("total", 0)
    total_pages = -(-total // PAGE_SIZE)   # ceiling division
    print(f"    Total properties : {total:,}")
    print(f"    Page size        : {PAGE_SIZE}")
    print(f"    Pages to fetch   : {total_pages:,}\n")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()

        # Write page 1 results (already fetched)
        for prop in first.get("results", []):
            writer.writerow(parse_property(prop))

        fetched = len(first.get("results", []))

        # ── Remaining pages ──────────────────────────────────────────────────
        for page in range(2, total_pages + 1):
            time.sleep(DELAY)
            data = fetch_page(session, page)
            results = data.get("results", [])

            for prop in results:
                writer.writerow(parse_property(prop))

            fetched += len(results)

            # Progress bar
            pct = fetched / total * 100
            bar = "█" * int(pct // 2) + "░" * (50 - int(pct // 2))
            sys.stdout.write(
                f"\r  [{bar}] {pct:5.1f}%  {fetched:,}/{total:,} properties  (page {page}/{total_pages})"
            )
            sys.stdout.flush()

    elapsed = (datetime.now() - start_time).total_seconds()
    print(f"\n\n {fetched:,} properties saved to '{OUTPUT_CSV}'")
    print(f"    Time elapsed: {elapsed:.1f}s")


if __name__ == "__main__":
    scrape_all()

    Nawy scraper started at 23:09:16
    Output → nawy_properties_full.csv

    Total properties : 18,187
    Page size        : 50
    Pages to fetch   : 364

  [████████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  40.1%  7,300/18,187 properties  (page 146/364)